In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-3-2-cluster-GO

Summarize Gene Ontology enrichment for trajectory clusters.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


end

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
from tqdm import tqdm
import textwrap
import logging



INPUT_CLUSTER_DIR = input_path('2-8.3-shanda/1-feature/10-2-0-Gene_Trajectory_Clusters_Annotated/')

OUTPUT_GO_DIR = output_path('2-8.3-shanda/1-feature/1-figure/0-3-result-4-nonlinear/2-cluster-GO')



TOP_N_TERMS = 8  

ENRICH_DB = 'GO_Biological_Process_2023' 

os.makedirs(OUTPUT_GO_DIR, exist_ok=True)


plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
sns.set_theme(style="ticks", rc={'axes.edgecolor': 'black'})

def normalize_gene_name_mouse(name: str) -> str:
    """Normalize mouse gene symbols to initial capital and lowercase remainder (GAPDH -> Gapdh)."""
    if not isinstance(name, str) or not name.strip():
        return str(name)
    name = name.strip()
    return name[0].upper() + name[1:].lower()

print(f"🚀 开始进行小鼠组织的 GO 富集分析与气泡图绘制 (紧凑布局版)...")


cluster_files = [f for f in os.listdir(INPUT_CLUSTER_DIR) if f.endswith('_Cluster_Assignments.csv')]

for c_file in tqdm(cluster_files, desc="Processing GO Enrichment"):
    tissue_name = c_file.replace('_Cluster_Assignments.csv', '')
    print(f"\n👉 正在处理组织: {tissue_name}")
    

    df_clusters = pd.read_csv(os.path.join(INPUT_CLUSTER_DIR, c_file))
    df_clusters.columns = ['Gene', 'Cluster']
    
    all_enrich_results = []
    

    unique_clusters = sorted(df_clusters['Cluster'].unique())
    for cluster_id in unique_clusters:

        raw_gene_list = df_clusters[df_clusters['Cluster'] == cluster_id]['Gene'].tolist()
        
        if len(raw_gene_list) < 5:
            print(f"  - Cluster {cluster_id + 1} 基因数太少({len(raw_gene_list)}), 跳过富集。")
            continue
            

        mouse_gene_list = [normalize_gene_name_mouse(g) for g in raw_gene_list]
            
        print(f"  - 正在富集 Cluster {cluster_id + 1} (包含 {len(mouse_gene_list)} 个基因)...")
        

        max_retries = 3
        for attempt in range(max_retries):
            try:

                time.sleep(3) 
                

                enr = gp.enrichr(gene_list=mouse_gene_list,
                                 gene_sets=ENRICH_DB,
                                 organism='mouse', 
                                 outdir=None,      
                                 no_plot=True)     
                
                res_df = enr.results

                res_df = res_df[res_df['P-value'] < 0.05].copy()
                
                if not res_df.empty:

                    res_df['Count'] = res_df['Overlap'].apply(lambda x: int(x.split('/')[0]))
                    res_df['Cluster'] = f"Cluster {cluster_id + 1}"
                    

                    top_terms = res_df.sort_values('P-value').head(TOP_N_TERMS)
                    all_enrich_results.append(top_terms)
                
                break
                
            except Exception as e:
                error_msg = str(e)
                if attempt < max_retries - 1:
                    print(f"    ⚠️ 网络连接异常，等待 5 秒后进行第 {attempt + 2} 次重试...")
                    time.sleep(5)
                else:
                    print(f"    ❌ Cluster {cluster_id + 1} 最终富集失败: {error_msg}")
        # ==============================================================
            
    if not all_enrich_results:
        print(f"  ⚠️ {tissue_name} 没有任何 Cluster 得到显著富集结果。")
        continue
        

    final_plot_df = pd.concat(all_enrich_results, ignore_index=True)
    

    final_plot_df['Term_Clean'] = final_plot_df['Term'].apply(lambda x: x.split(' (GO:')[0])

    final_plot_df['-log10(P-value)'] = -np.log10(final_plot_df['P-value'])
    

    csv_out = os.path.join(OUTPUT_GO_DIR, f"{tissue_name}_GO_Enrichment_Summary.csv")
    final_plot_df.to_csv(csv_out, index=False)
    


    dynamic_width = max(4.5, len(unique_clusters) * 0.8)
    dynamic_height = max(5, len(final_plot_df['Term_Clean'].unique()) * 0.35)
    
    plt.figure(figsize=(dynamic_width, dynamic_height))
    
    scatter = sns.scatterplot(data=final_plot_df, 
                              x='Cluster', 
                              y='Term_Clean',
                              size='Count', 
                              hue='-log10(P-value)', 
                              sizes=(50, 400), 
                              palette='viridis', 
                              edgecolor='black',
                              linewidth=0.5)
    

    plt.title(f"{tissue_name} - Gene Ontology Enrichment", fontsize=14, pad=15, fontweight='bold')
    plt.xlabel("")
    plt.ylabel("")
    

    plt.xticks(rotation=45, ha='right', fontsize=11)
    

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title_fontsize=10)
    

    plt.grid(True, linestyle='--', alpha=0.5, zorder=0)
    scatter.set_axisbelow(True)
    

    pdf_out = os.path.join(OUTPUT_GO_DIR, f"{tissue_name}_GO_BubblePlot.pdf")
    plt.savefig(pdf_out, bbox_inches='tight', transparent=True)
    

    plt.close()

print(f"\n✅ 完成！所有小鼠组织的 GO 富集紧凑版气泡图已保存在:\n{OUTPUT_GO_DIR}")